reference: https://github.uio.no/natallid/FYS-STK3155_MachineLearning/blob/main/Code/Project_1.ipynb

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import os # for saving figures
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import PolynomialFeatures
import matplotlib.pyplot as plt
from numpy.linalg import inv
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
import statistics
from sklearn.utils import resample
import warnings

In [2]:
# To avoid lengthy RunTime warnings (for overflow cases)
warnings.filterwarnings('ignore')

In [3]:
# Setting seed for reproducibility
np.random.seed(42)
n = 200

# Creating a grid of n=200 points drawn from the random distribution
x = np.random.uniform(-1, 1, n)

# Setting up Runge's function, 1 normal and 1 with added noise
y_noise = 1.0 / (1 + 25 * x**2)+ np.random.normal(0, 0.1, x.shape)

# Calculating sample statistics
x_mean = np.mean(x)
x_variance = np.std(x, ddof=1) #sample variance

#Scaling x values for a visual comparison
x_st = (x - x_mean) / x_variance

In [4]:
# Now we split the dataset for training and testing (using the standardized x values) and noisy Runge
X_train, X_test, y_train_noise, y_test_noise = train_test_split(x_st, y_noise, test_size=0.2, random_state = 42)


In [5]:
from sklearn.preprocessing import PolynomialFeatures
np.random.seed(42)


def gradient_descent_with_momentum(X_train, y_train, degree, lambda_, n_iter, step_size, momentum=0.3, tolerance=None):
    """A function implementing gradient descent with momentum, if OLS, lambda=0."""

    # Creating the design matrix with PolynomialFeatures
    poly = PolynomialFeatures(degree=degree, include_bias=True)
    X_poly = poly.fit_transform(X_train)

    # Finding the dimensions and setting up theta vector
    n_samples, n_features = X_poly.shape
    theta = np.zeros(n_features)
    #np.random.randn(n_features) * 0.1

    # Initializing the momentum
    velocity = np.zeros(n_features)

    # Initializing empty lists for metrics
    theta_history = []
    loss_history = []

    for i in range(n_iter):
‡
        # Predicting and calculating error
        y_pred = X_poly @ theta
        error = y_pred - y_train

        # Using the analytical expression for the gradient (if OLS, lambda=0)
        gradient = 2*(X_poly.T @ error) / n_samples + 2 * lambda_ * theta

        # Checking for early convergence
        if tolerance is not None and np.linalg.norm(gradient) < tolerance:
            print(f"Converged at iteration {i+1}")
            break 

        # Updating velocity and parameters
        velocity = momentum * velocity - step_size * gradient
        theta += velocity

        # Calculating metrics
        mse = np.mean(error**2)
        loss = mse + lambda_ * np.sum(theta**2)

        # Storing metrics (using .copy to add a list, not a pointer)
        theta_history.append(theta.copy())
        loss_history.append(loss)

    return theta_history, loss_history, theta, poly

In [6]:
# Setting up hyperparameters for grid search 
optimizers = [gradient_descent_with_momentum]
degrees = np.arange(1, 16)
lambdas= [0]
learning_rates = [0.001, 0.01, 0.05, 0.1, 0.5]

# Tracking results
best_mse = float("inf")
best_params = {}
all_results = []


# Performing grid search
for optimizer in optimizers:
    optimizer_name = optimizer.__name__

    print(f"Testing optimizer: {optimizer_name} \n---------------------------------")

    for degree in degrees:
        for lambda_ in lambdas:
            for lr in learning_rates:

                # Training the model
                theta_history, loss_history, theta, poly = optimizer(
                    X_train.reshape(-1, 1), y_train_noise, degree=degree, lambda_=lambda_, n_iter=1000, step_size=lr, tolerance=1e-10
                )
                # Evaluating on the test set
                X_test_poly = poly.transform(X_test.reshape(-1, 1))
                y_test_pred = X_test_poly @ theta
                test_mse = np.mean((y_test_pred - y_test_noise)**2)

                # Store results
                result = {
                    "optimizer": optimizer_name,
                    "degree": degree,
                    "lambda": lambda_,
                    "learning_rate": lr,
                    "test_mse": test_mse
                }
                all_results.append(result)

                # Checking whether the mse is the best so far
                if test_mse < best_mse:
                    best_mse = test_mse
                    best_params = result.copy()

Testing optimizer: gradient_descent_with_momentum 
---------------------------------
Converged at iteration 766
Converged at iteration 135
Converged at iteration 48
Converged at iteration 36
Converged at iteration 507
Converged at iteration 243
Converged at iteration 53
Converged at iteration 550


In [7]:
# Displaying grid search results
print(f"{'-'*50}")
print(f"\nBest Test MSE: {best_mse:.6f}")
print(f"Best Optimizer: {best_params['optimizer']}")
print(f"Best Degree: {best_params['degree']}")
print(f"Best Lambda: {best_params['lambda']}")
print(f"Best Learning Rate: {best_params['learning_rate']}")

# Asserting that the number of configurations
print(len(all_results))

# Using DataFrame from pandas to visualize
import pandas as pd
df_results = pd.DataFrame(all_results)
print(f"\nTop 10 Ridge regression configurations:")
print(df_results.nsmallest(10, 'test_mse'))

--------------------------------------------------

Best Test MSE: 0.025996
Best Optimizer: gradient_descent_with_momentum
Best Degree: 5
Best Lambda: 0
Best Learning Rate: 0.01
75

Top 10 Ridge regression configurations:
                         optimizer  degree  lambda  learning_rate  test_mse
21  gradient_descent_with_momentum       5       0           0.01  0.025996
16  gradient_descent_with_momentum       4       0           0.01  0.026458
17  gradient_descent_with_momentum       4       0           0.05  0.030757
18  gradient_descent_with_momentum       4       0           0.10  0.030822
22  gradient_descent_with_momentum       5       0           0.05  0.030995
6   gradient_descent_with_momentum       2       0           0.01  0.034537
7   gradient_descent_with_momentum       2       0           0.05  0.034543
8   gradient_descent_with_momentum       2       0           0.10  0.034543
9   gradient_descent_with_momentum       2       0           0.50  0.034543
26  gradient_desce